# Geração de dados mockados — localização e status de motoristas

Objetivo: gerar ~10 registros por driver (10 drivers, ~100 linhas) para
exercitar point-in-time joins no historical features do Feast, e escrever
o resultado como uma Delta table local (offline store).

Dados 100% sintéticos, sem qualquer relação com clientes reais.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from deltalake import DeltaTable, write_deltalake

## Parâmetros

In [2]:
N_DRIVERS = 10
N_EVENTS_PER_DRIVER = 10
BASE_TIME = datetime.now(timezone.utc) - timedelta(days=10)
STATUS_CHOICES = ["available", "en_route", "busy", "offline"]
BASE_LAT, BASE_LON = -23.5505, -46.6333  # São Paulo, apenas referência de mock

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
DELTA_PATH = REPO_ROOT / "data" / "offline_store" / "driver_stats"

## Geração dos eventos por driver

Timestamps espaçados de forma não uniforme ao longo de 10 dias, para dar
margem a timestamps "as of" que caem entre eventos no historical features.
`created_timestamp` simula chegada tardia do dado (alguns segundos depois
do `event_timestamp`), usado para desduplicação no point-in-time join.

In [3]:
rng = np.random.default_rng(seed=42)
rows = []
for driver_id in range(1000, 1000 + N_DRIVERS):
    t = BASE_TIME
    lat = BASE_LAT + rng.normal(0, 0.05)
    lon = BASE_LON + rng.normal(0, 0.05)
    for _ in range(N_EVENTS_PER_DRIVER):
        t = t + timedelta(minutes=int(rng.integers(30, 600)))
        lat += rng.normal(0, 0.01)
        lon += rng.normal(0, 0.01)
        event_ts = t
        created_ts = event_ts + timedelta(seconds=int(rng.integers(1, 120)))
        rows.append(
            {
                "driver_id": driver_id,
                "event_timestamp": event_ts,
                "created_timestamp": created_ts,
                "latitude": float(lat),
                "longitude": float(lon),
                "status": str(rng.choice(STATUS_CHOICES)),
            }
        )

df = pd.DataFrame(rows)
df["driver_id"] = df["driver_id"].astype("int64")
df["latitude"] = df["latitude"].astype("float32")
df["longitude"] = df["longitude"].astype("float32")
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
df["created_timestamp"] = pd.to_datetime(df["created_timestamp"], utc=True)

## Validação básica antes de escrever

In [4]:
assert df.shape[0] == N_DRIVERS * N_EVENTS_PER_DRIVER
assert df["driver_id"].nunique() == N_DRIVERS
assert set(df["status"].unique()) <= set(STATUS_CHOICES)
print(df.shape)
df.head()

(100, 6)


,driver_id,event_timestamp,created_timestamp,latitude,longitude,status
0,1000,2026-08-06 21:27:09.181619+00:00,2026-08-06 21:28:52.181619+00:00,-23.525858,-46.704811,busy
1,1000,2026-08-07 07:13:09.181619+00:00,2026-08-07 07:14:11.181619+00:00,-23.524580,-46.707973,available
2,1000,2026-08-07 15:41:09.181619+00:00,2026-08-07 15:42:03.181619+00:00,-23.515785,-46.700195,offline
3,1000,2026-08-07 22:18:09.181619+00:00,2026-08-07 22:19:03.181619+00:00,-23.504513,-46.695518,available
4,1000,2026-08-07 23:40:09.181619+00:00,2026-08-07 23:41:15.181619+00:00,-23.514103,-46.686733,en_route


## Escrita como Delta table local

In [5]:
DELTA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_deltalake(str(DELTA_PATH), df, mode="overwrite")
print(f"Delta table escrita em: {DELTA_PATH}")

Delta table escrita em: c:\Users\USER\mlops\feature-store-playground\data\offline_store\driver_stats


## Validação de releitura

In [6]:
dt = DeltaTable(str(DELTA_PATH))
df_check = dt.to_pandas()
print(df_check.shape, df_check["driver_id"].nunique())
df_check.sort_values(["driver_id", "event_timestamp"]).head(10)

(100, 6) 10


,driver_id,event_timestamp,created_timestamp,latitude,longitude,status
0,1000,2026-08-06 21:27:09.181619+00:00,2026-08-06 21:28:52.181619+00:00,-23.525858,-46.704811,busy
1,1000,2026-08-07 07:13:09.181619+00:00,2026-08-07 07:14:11.181619+00:00,-23.524580,-46.707973,available
2,1000,2026-08-07 15:41:09.181619+00:00,2026-08-07 15:42:03.181619+00:00,-23.515785,-46.700195,offline
3,1000,2026-08-07 22:18:09.181619+00:00,2026-08-07 22:19:03.181619+00:00,-23.504513,-46.695518,available
4,1000,2026-08-07 23:40:09.181619+00:00,2026-08-07 23:41:15.181619+00:00,-23.514103,-46.686733,en_route
5,1000,2026-08-08 06:10:09.181619+00:00,2026-08-08 06:10:18.181619+00:00,-23.515951,-46.693542,offline
6,1000,2026-08-08 10:54:09.181619+00:00,2026-08-08 10:55:56.181619+00:00,-23.520235,-46.697063,en_route
7,1000,2026-08-08 15:50:09.181619+00:00,2026-08-08 15:51:38.181619+00:00,-23.516581,-46.692936,busy
8,1000,2026-08-09 01:05:09.181619+00:00,2026-08-09 01:06:38.181619+00:00,-23.520643,-46.698059,offline
9,1000,2026-08-09 05:06:09.181619+00:00,2026-08-09 05:07:05.181619+00:00,-23.514484,-46.686771,available
